### Resolving $k=2$ When $4 \mid n$ 

As detailed in Section 3.2 of the manuscript, the methods used to bound solutions for larger even values of $k$ cannot be applied when $k=2$ because $T_2(x) = 1$. Instead, the generalized equation simplifies to:
$$x(x+1)(2x+1) = 6p^\alpha y^4$$

Because the factors $x$, $x+1$, and $2x+1$ are pairwise coprime, the prime $p$ can divide at most one of them. By setting $x+1 = Aa^4$, $x = Bb^4$, and $2x+1 = Cc^4$, we obtain three families of binomial Thue equations depending on which factor $p$ divides.

The script below generates these three families of equations, enforcing the constraint that the coefficient $C$ must be odd (since it corresponds to $2x+1$), and unconditionally solves them using PARI/GP. The output verifies that the only valid solutions restrict the variables to $|ab| \in \{0, 1\}$, isolating $(x,y,k,n,p,\alpha) = (2,1,2,n,5,1)$ as the sole near-solution.

In [ ]:
# =============================================================================
# Thue Equation Resolutions for k = 2 and 4 | n
# =============================================================================

R.<X, Y> = PolynomialRing(QQ)

def solve_thue(f, m):
    """Unconditionally solves the Thue equation f = m without GRH."""
    assert f.is_homogeneous()
    parithueinit = gp.thueinit(f.subs({f.variables()[1]: 1}), flag=1)
    return gp.thue(parithueinit, m).sage()

def resolve_k2_n4_cases():
    """
    Solves the three families of binomial Thue equations arising from 
    k = 2 and n divisible by 4, filtering for valid coprime pairs.
    """
    print("Resolving Thue equations for k = 2 and 4 | n...")
    print("-" * 75)
    
    # Generate all pairs (u, v) such that u * v divides 6
    div_pairs = [(u, v) for u in divisors(6) for v in divisors(6 / u)]
    
    print("Case 1: p | 2x + 1  =>  Aa^4 - Bb^4 = 1  (with AB | 6)")
    for A, B in div_pairs:
        sols = solve_thue(A * X^4 - B * Y^4, 1)
        print(f"  A = {A}, B = {B}  --->  Solutions (a, b): {sols}")
        
    print("\nCase 2: p | x + 1   =>  Cc^4 - 2Bb^4 = 1  (with BC | 6)")
    # C corresponds to 2x+1, so C must be odd. 
    # We use a set to ensure we only test unique, valid (B, C) pairs.
    case2_pairs = set()
    for u, v in div_pairs:
        B, C = (v, u) if v % 2 == 0 else (u, v)
        if C % 2 != 0:
            case2_pairs.add((B, C))
            
    for B, C in sorted(case2_pairs):
        sols = solve_thue(C * X^4 - 2 * B * Y^4, 1)
        print(f"  B = {B}, C = {C}  --->  Solutions (c, b): {sols}")
        
    print("\nCase 3: p | x       =>  2Aa^4 - Cc^4 = 1  (with AC | 6)")
    # Similarly, C corresponds to 2x+1, so C must be odd.
    case3_pairs = set()
    for u, v in div_pairs:
        A, C = (v, u) if v % 2 == 0 else (u, v)
        if C % 2 != 0:
            case3_pairs.add((A, C))
            
    for A, C in sorted(case3_pairs):
        sols = solve_thue(2 * A * X^4 - C * Y^4, 1)
        print(f"  A = {A}, C = {C}  --->  Solutions (a, c): {sols}")

# Execute the solver
resolve_k2_n4_cases()

Resolving Thue equations for k = 2 and 4 | n...
---------------------------------------------------------------------------
Case 1: p | 2x + 1  =>  Aa^4 - Bb^4 = 1  (with AB | 6)
  A = 1, B = 1  --->  Solutions (a, b): [[-1, 0], [1, 0]]
  A = 1, B = 2  --->  Solutions (a, b): [[-1, 0], [1, 0]]
  A = 1, B = 3  --->  Solutions (a, b): [[-1, 0], [1, 0]]
  A = 1, B = 6  --->  Solutions (a, b): [[-1, 0], [1, 0]]
  A = 2, B = 1  --->  Solutions (a, b): [[-1, -1], [-1, 1], [1, -1], [1, 1]]
  A = 2, B = 3  --->  Solutions (a, b): []
  A = 3, B = 1  --->  Solutions (a, b): []
  A = 3, B = 2  --->  Solutions (a, b): [[-1, -1], [-1, 1], [1, -1], [1, 1]]
  A = 6, B = 1  --->  Solutions (a, b): []

Case 2: p | x + 1   =>  Cc^4 - 2Bb^4 = 1  (with BC | 6)
  B = 1, C = 1  --->  Solutions (c, b): [[-1, 0], [1, 0]]
  B = 1, C = 3  --->  Solutions (c, b): [[-1, -1], [-1, 1], [1, -1], [1, 1]]
  B = 2, C = 1  --->  Solutions (c, b): [[-1, 0], [1, 0]]
  B = 2, C = 3  --->  Solutions (c, b): []
  B = 3, C = 

### Resolving $k=4$ When $n$ is Even

As detailed in Section 3.3 of the manuscript, when $n$ is even, we can reduce the problem to finding integral points on a finite number of elliptic curves. The methodology depends on whether the unknown prime $p$ divides the polynomial $T_4(x) = 3x^2 + 3x - 1$.

**Case 1: $p \mid T_4(x)$**
Dividing the general equation by $T_4(x)$ leads to elliptic curves of the form $Y^2 = X^3 + 3cX^2 + 2c^2X$, where $c$ is a divisor of $210$. We find all integral points on these curves subject to the constraints $2c \mid X$ and $2c^2 \mid Y$.

**Case 2: $p \nmid T_4(x)$**
In this case, $p$ must divide exactly one of the linear factors $x, x+1,$ or $2x+1$. We divide by the product of any two of these factors to obtain three new families of elliptic curves, again with $c \mid 210$.

The script below automates the extraction of integral points across all these curves, filters for valid candidate $x$-values, and then evaluates $S_4(x)$ to determine which candidates yield near-solutions to Schäffer's equation of the form $p^\alpha y^n$.

In [ ]:
# =============================================================================
# Elliptic Curve Search for k = 4 and n Even
# =============================================================================

var('x')

def S(k):
    """Returns S_k(x) as a polynomial in x using Faulhaber's formula."""
    return (1 / (k + 1)) * sum(
        binomial(k + 1, m) * ((-1)^m) * bernoulli(m) * x^(k - m + 1) 
        for m in [0..k]
    )

def find_p_divides_T4():
    """
    Finds valid x-values from the elliptic curves when p divides T_4(x).
    Returns a set of valid x candidates.
    """
    valid_x = set()
    D = divisors(210)
    
    for c in D:
        Ec4 = EllipticCurve([0, 3*c, 0, 2*c^2, 0])
        pts = Ec4.integral_points()
        
        for pt in pts:
            X, Y = pt[0], pt[1]
            if X > 0 and X % (2*c) == 0 and Y % (2*c^2) == 0:
                x_val = X / (2*c)
                w_val = abs(Y) / (2*c^2)
                valid_x.add(x_val)
                print(f"Case 1 (p | T_4): c={c:<3} -> Found candidate x={x_val}, w={w_val}")
                
    return valid_x

def find_p_not_divides_T4():
    """
    Finds valid x-values from the three elliptic curve families when p 
    does not divide T_4(x). Returns a set of valid x candidates.
    """
    valid_x = set()
    D = divisors(210)
    
    for c in D:
        Ec41 = EllipticCurve([0, 3*c, 0, -3*c^2, 0])
        Ec42 = EllipticCurve([0, 6*c, 0, 6*c^2, -9*c^3])
        Ec43 = EllipticCurve([0, 9*c, 0, 6*c^2, -36*c^3])
        
        pts1 = Ec41.integral_points()
        pts2 = Ec42.integral_points()
        pts3 = Ec43.integral_points()
        
        # Curve 1 Checks
        for pt in pts1:
            X, Y = pt[0], pt[1]
            if X > 0 and X % (3*c) == 0 and Y % (3*c^2) == 0:
                x_val = X / (3*c)
                w_val = abs(Y) / (3*c^2)
                valid_x.add(x_val)
                print(f"Case 2 (Curve 1): c={c:<3} -> Found candidate x={x_val}, w={w_val}")
                
        # Curve 2 Checks
        for pt in pts2:
            X, Y = pt[0], pt[1]
            if X > 0 and X % (3*c) == 0 and Y % (3*c^2) == 0:
                x_val = X / (3*c)
                w_val = abs(Y) / (3*c^2)
                valid_x.add(x_val)
                print(f"Case 2 (Curve 2): c={c:<3} -> Found candidate x={x_val}, w={w_val}")
                
        # Curve 3 Checks (Requires odd c)
        if c % 2 == 1:
            for pt in pts3:
                X, Y = pt[0], pt[1]
                if X > 0 and X % (6*c) == 0 and Y % (6*c^2) == 0:
                    x_val = X / (6*c)
                    w_val = abs(Y) / (6*c^2)
                    valid_x.add(x_val)
                    print(f"Case 2 (Curve 3): c={c:<3} -> Found candidate x={x_val}, w={w_val}")
                    
    return valid_x

def verify_S4_solutions(candidate_x_values):
    """
    Evaluates S_4(x) for all candidate x-values and displays the factorization
    to verify solutions of the form p^alpha * y^n.
    """
    print("\n" + "=" * 60)
    print("Verifying Candidates against S_4(x) = p * y^2")
    print("=" * 60)
    
    # Sort for ascending output
    for i in sorted(list(candidate_x_values)):
        s_val = ZZ(S(4)(x=i))
        factors = factor(s_val)
        print(f"x = {str(i):<4} | S_4({i}) = {str(factors)}")

# =============================================================================
# Execution Pipeline
# =============================================================================

print("Searching for integral points on elliptic curves for k = 4...")
print("-" * 60)

# Execute searches
candidates_case1 = find_p_divides_T4()
candidates_case2 = find_p_not_divides_T4()

# Combine unique candidates from both cases
all_candidates = candidates_case1.union(candidates_case2)

# Run verifications
verify_S4_solutions(all_candidates)

Searching for integral points on elliptic curves for k = 4...
------------------------------------------------------------
Case 1 (p | T_4): c=5   -> Found candidate x=4, w=6
Case 1 (p | T_4): c=6   -> Found candidate x=1, w=1
Case 1 (p | T_4): c=6   -> Found candidate x=24, w=70
Case 1 (p | T_4): c=21  -> Found candidate x=3, w=2
Case 1 (p | T_4): c=30  -> Found candidate x=2, w=1
Case 1 (p | T_4): c=210 -> Found candidate x=7, w=2
Case 1 (p | T_4): c=210 -> Found candidate x=840, w=2378
Case 2 (Curve 1): c=5   -> Found candidate x=1, w=1
Case 2 (Curve 3): c=5   -> Found candidate x=3, w=7
Case 2 (Curve 3): c=5   -> Found candidate x=283, w=5229
Case 2 (Curve 2): c=10  -> Found candidate x=1, w=1
Case 2 (Curve 3): c=15  -> Found candidate x=1, w=1
Case 2 (Curve 1): c=30  -> Found candidate x=6, w=5
Case 2 (Curve 2): c=35  -> Found candidate x=3, w=2
Case 2 (Curve 2): c=35  -> Found candidate x=6, w=5
Case 2 (Curve 1): c=105 -> Found candidate x=3, w=1

Verifying Candidates against S_4

### Resolving $k=6$ When $n$ is Even

When $k=6$, we have $T_6(x) = 3(x^2+x)^2 - 3(x^2+x) + 1$. The method for bounding solutions for even $n$ again depends on whether the unknown prime $p$ divides $T_6(x)$.

**Case 1: $p \nmid T_6(x)$**
Dividing the general equation by the linear factors $x(x+1)(2x+1)$ leads to $16T_6(x) = 3v^4 - 18v^2 + 31 = c w^2$, where $v = 2x+1$ and $c \mid 217$. By multiplying both sides by $v^2$ and substituting $X = v^2$, we avoid a hyperelliptic curve and reduce the problem to finding integral points on the elliptic curves $Y^2 = X^3 - 18cX^2 + 31c^2X$. We then filter the integral points to check if $X$ is a perfect square.

**Case 2: $p \mid T_6(x)$**
In this case, dividing by $T_6(x)$ leaves the linear factors $x(x+1)(2x+1) = c w^2$ for $c \mid 1302$. Multiplying by $c^3$ and applying a change of variables yields a second family of elliptic curves: $Y^2 = X^3 + 3cX^2 + 2c^2X$.

The script below calculates the integral points across both families of curves, handles PARI/GP rank computation, reverses the changes of variables, and tests any valid integer $x$ candidates against $S_6(x)$ to isolate solutions.

In [ ]:
# =============================================================================
# Elliptic Curve Search for k = 6 and n Even
# =============================================================================

var('x')

def S(k):
    """Returns S_k(x) as a polynomial in x using Faulhaber's formula."""
    return (1 / (k + 1)) * sum(
        binomial(k + 1, m) * ((-1)^m) * bernoulli(m) * x^(k - m + 1) 
        for m in [0..k]
    )

def find_k6_case1_candidates():
    """
    Finds valid x-values from the elliptic curves when p does not divide T_6.
    Uses the substitution X = v^2 to form a cubic elliptic curve.
    """
    valid_x = set()
    divs_217 = divisors(217)
    
    # f(X) = 3X^3 - 18X^2 + 31X  (where X = v^2)
    f = 3 * x^3 - 18 * x^2 + 31 * x
    g = expand(3^2 * f(x = x / 3))
    
    for c in divs_217:
        gc = c^3 * expand(g(x = x / c))
        gccoefs = gc.coefficients(sparse=False)
        Ec = EllipticCurve([0, Integer(gccoefs[2]), 0, Integer(gccoefs[1]), Integer(gccoefs[0])])
        
        try:
            Ec.rank(algorithm='pari')
            pts = Ec.integral_points()
        except RuntimeError:
            # Fallback if Pari fails to compute the rank easily
            badcrank = Ec.rank(only_use_mwrank=False, algorithm='pari')
            pts = Ec.torsion_points() if badcrank == 0 else Ec.integral_points()
                
        for pt in pts:
            X, Y = pt[0], pt[1]
            # Check divisibility constraints from the change of variables
            if Y != 0 and Y % (3 * c^2) == 0 and X % (3 * c) == 0 and X % 2 == 1:
                X_orig = X / c
                v_squared = X_orig / 3
                
                # We substituted X = v^2 earlier, so we must verify it is a perfect square
                if v_squared > 0 and sqrt(v_squared) in ZZ:
                    v = sqrt(v_squared)
                    x_val = (v - 1) / 2
                    if x_val in ZZ:
                        valid_x.add(x_val)
                        
    return valid_x

def find_k6_case2_candidates():
    """
    Finds valid x-values from the elliptic curves when p divides T_6.
    """
    valid_x = set()
    divs_1302 = divisors(1302)
    
    f2 = x * (x + 1) * (2 * x + 1)
    g2 = expand(2^2 * f2(x = x / 2))
    
    for c in divs_1302:
        hc = c^3 * expand(g2(x = x / c))
        hcoefs = hc.coefficients(sparse=False)
        Ec = EllipticCurve([0, Integer(hcoefs[2]), 0, Integer(hcoefs[1]), Integer(hcoefs[0])])
        
        try:
            Ec.rank(algorithm='pari', pari_effort=25)
            pts = Ec.integral_points()
        except RuntimeError:
            badcrank = Ec.rank(only_use_mwrank=False, algorithm='pari', pari_effort=25)
            pts = Ec.torsion_points() if badcrank == 0 else Ec.integral_points()
                
        for pt in pts:
            X, Y = pt[0], pt[1]
            # Check divisibility constraints from the change of variables
            if Y != 0 and Y % (2 * c^2) == 0 and X > 0 and X % (2 * c) == 0:
                x_val = X / (2 * c)
                if x_val in ZZ:
                    valid_x.add(x_val)
                    
    return valid_x

def verify_S6_solutions(candidate_x_values):
    """
    Evaluates S_6(x) for all candidate x-values to determine if the squarefree
    part is prime, isolating near-solutions of the form p^alpha * y^n.
    """
    print("\n" + "=" * 70)
    print("Verifying Candidates against S_6(x) = p * y^2")
    print("=" * 70)
    
    for x_val in sorted(list(candidate_x_values)):
        s_val = ZZ(S(6)(x=x_val))
        
        # Omit trivial zero cases
        if s_val == 0:
            continue
            
        sqf_part = s_val.squarefree_part()
        
        if sqf_part.is_prime():
            y_val = sqrt(s_val / sqf_part)
            print(f"VALID SOLUTION: x = {str(x_val):<3} | y = {str(y_val):<5} | p = {str(sqf_part)}")
        else:
            print(f"Dismissed: x = {str(x_val):<3} | Squarefree part ({str(sqf_part)}) is not prime.")

# =============================================================================
# Execution Pipeline
# =============================================================================

print("Searching for integral points on elliptic curves for k = 6...")
print("-" * 70)

# Execute searches
candidates_case1 = find_k6_case1_candidates()
candidates_case2 = find_k6_case2_candidates()

# Combine unique candidates from both cases
all_candidates = candidates_case1.union(candidates_case2)

# Run verifications
if all_candidates:
    verify_S6_solutions(all_candidates)
else:
    print("\nNo integer candidates found.")

Searching for integral points on elliptic curves for k = 6...
----------------------------------------------------------------------

Verifying Candidates against S_6(x) = p * y^2
Dismissed: x = 1   | Squarefree part (1) is not prime.
Dismissed: x = 3   | Squarefree part (794) is not prime.
Dismissed: x = 24  | Squarefree part (7547407) is not prime.
Dismissed: x = 31  | Squarefree part (274277181) is not prime.


### Resolving $k=8$ When $n$ is Even

As detailed in Section 3.3 of the manuscript, when $k=8$ and $n$ is even, we use the substitution $u = x(x+1)$ to write $T_8(x) = 5u^3 - 10u^2 + 9u - 3$. The method for bounding solutions depends on whether the unknown prime $p$ divides $T_8(x)$.

**Case 1: $p \nmid T_8(x)$**
Dividing the general equation by the linear factors $x(x+1)(2x+1)$ gives $T_8(x) = c w^2$, where $c \mid 3810$, multiplying both sides by $25c^3$ and applying the substitution $X = 5cu$, we obtain the family of elliptic curves $Y^2 = X^3 - 10cX^2 + 45c^2X - 75c^3$. We find all integral points subject to the divisibility constraints of the substitution, and verify if the resulting $u$ yields an integer $x$. (If SageMath struggles to compute the rank of this curve, we use the substitution $V = (2x+1)^2$ to form a secondary fallback elliptic curve).

**Case 2: $p \mid T_8(x)$**
Dividing by $T_8(x)$ leaves the linear factors $x(x+1)(2x+1) = c w^2$ for $c \mid 3810$. Multiplying by $4c^3$ and applying a change of variables yields a second family of elliptic curves: $Y^2 = X^3 + 3cX^2 + 2c^2X$.

The script below automates the extraction of integral points across both families of curves, handles the fallback curve generation, and evaluates valid integer $x$ candidates against $S_8(x)$ to isolate near-solutions to Schäffer's conjecture.

In [ ]:
# =============================================================================
# Elliptic Curve Search for k = 8 and n Even
# =============================================================================

var('x')

def S(k):
    """Returns S_k(x) as a polynomial in x using Faulhaber's formula."""
    return (1 / (k + 1)) * sum(
        binomial(k + 1, m) * ((-1)^m) * bernoulli(m) * x^(k - m + 1) 
        for m in [0..k]
    )

def find_k8_case1_candidates():
    """
    Finds valid x-values from the elliptic curves when p does not divide T_8.
    Uses the substitution X = 5cu to form the primary elliptic curve.
    """
    valid_x = set()
    divs_3810 = divisors(3810)
    
    # f(u) = 5u^3 - 10u^2 + 9u - 3
    f = 5 * x^3 - 10 * x^2 + 9 * x - 3
    g = expand(5^2 * f(x = x / 5))
    
    for c in divs_3810:
        gc = c^3 * expand(g(x = x / c))
        gccoefs = gc.coefficients(sparse=False)
        Ec = EllipticCurve([0, Integer(gccoefs[2]), 0, Integer(gccoefs[1]), Integer(gccoefs[0])])
        
        try:
            Ec.rank(algorithm='pari')
            pts = Ec.integral_points()
            
            for pt in pts:
                X, Y = pt[0], pt[1]
                if Y != 0 and Y % (5 * c^2) == 0 and X % (5 * c) == 0:
                    u = X / (5 * c)
                    if u > 0 and sqrt(4 * u + 1) in ZZ:
                        v = sqrt(4 * u + 1)
                        x_val = (v - 1) / 2
                        if x_val in ZZ:
                            valid_x.add(x_val)
                            
        except RuntimeError:
            # Fallback: Form the curve using V = v^2 = 4u + 1
            f1 = 5 * x^3 - 55 * x^2 + 239 * x - 381
            g1 = expand(5^2 * f1(x = x / 5))
            g1c = c^3 * expand(g1(x = x / c))
            g1ccoefs = g1c.coefficients(sparse=False)
            E1c = EllipticCurve([0, Integer(g1ccoefs[2]), 0, Integer(g1ccoefs[1]), Integer(g1ccoefs[0])])
            
            try:
                badcrank = E1c.rank(only_use_mwrank=False, algorithm='pari')
                pts = E1c.torsion_points() if badcrank == 0 else E1c.integral_points()
                
                for pt in pts:
                    X, Y = pt[0], pt[1]
                    if Y != 0 and Y % (5 * c^2) == 0 and X % (5 * c) == 0:
                        V = X / (5 * c)
                        if V > 0 and sqrt(V) in ZZ:
                            v = sqrt(V)
                            x_val = (v - 1) / 2
                            if x_val in ZZ:
                                valid_x.add(x_val)
                                
            except RuntimeError:
                print(f"Skipping c={c}: SageMath failed to compute rank on both curves.")
                
    return valid_x

def find_k8_case2_candidates():
    """
    Finds valid x-values from the elliptic curves when p divides T_8.
    """
    valid_x = set()
    divs_3810 = divisors(3810)
    
    f2 = x * (x + 1) * (2 * x + 1)
    g2 = expand(2^2 * f2(x = x / 2))
    
    for c in divs_3810:
        hc = c^3 * expand(g2(x = x / c))
        hcoefs = hc.coefficients(sparse=False)
        Ec = EllipticCurve([0, Integer(hcoefs[2]), 0, Integer(hcoefs[1]), Integer(hcoefs[0])])
        
        try:
            Ec.rank(algorithm='pari', pari_effort=25)
            pts = Ec.integral_points()
        except RuntimeError:
            badcrank = Ec.rank(only_use_mwrank=False, algorithm='pari', pari_effort=25)
            pts = Ec.torsion_points() if badcrank == 0 else Ec.integral_points()
                
        for pt in pts:
            X, Y = pt[0], pt[1]
            if Y != 0 and Y % (2 * c^2) == 0 and X > 0 and X % (2 * c) == 0:
                x_val = X / (2 * c)
                if x_val in ZZ:
                    valid_x.add(x_val)
                    
    return valid_x

def verify_S8_solutions(candidate_x_values):
    """
    Evaluates S_8(x) for all candidate x-values to isolate near-solutions 
    of the form p^\alpha * y^n.
    """
    print("\n" + "=" * 70)
    print("Verifying Candidates against S_8(x) = p * y^2")
    print("=" * 70)
    
    for x_val in sorted(list(candidate_x_values)):
        s_val = ZZ(S(8)(x=x_val))
        
        if s_val == 0:
            continue
            
        sqf_part = s_val.squarefree_part()
        
        if sqf_part.is_prime():
            y_val = sqrt(s_val / sqf_part)
            print(f"VALID SOLUTION: x = {str(x_val):<3} | y = {str(y_val):<5} | p = {sqf_part}")
        else:
            print(f"Dismissed: x = {str(x_val):<3} | Squarefree part ({sqf_part}) is not prime.")

# =============================================================================
# Execution Pipeline
# =============================================================================

print("Searching for integral points on elliptic curves for k = 8...")
print("-" * 70)

# Execute searches
candidates_case1 = find_k8_case1_candidates()
candidates_case2 = find_k8_case2_candidates()

# Combine unique candidates from both cases
all_candidates = candidates_case1.union(candidates_case2)

# Run verifications
if all_candidates:
    verify_S8_solutions(all_candidates)
else:
    print("\nNo integer candidates found.")

Searching for integral points on elliptic curves for k = 8...
----------------------------------------------------------------------

Verifying Candidates against S_8(x) = p * y^2
Dismissed: x = 1   | Squarefree part (1) is not prime.
VALID SOLUTION: x = 2   | y = 1     | p = 257
Dismissed: x = 4   | Squarefree part (72354) is not prime.
Dismissed: x = 24  | Squarefree part (1794008995) is not prime.


### Resolving $k=10$ When $n$ is Even

When $k=10$ and $n$ is even, we use the substitution $u = x(x+1)$ to write $T_{10}(x) = (u-1)(3u^3 - 7u^2 + 10u - 5)$. The solution depends on whether $p$ divides $T_{10}(x)$.

**Case 1: $p \nmid T_{10}(x)$**
Dividing the general equation by the linear factors and $u-1$ leaves $3u^3 - 7u^2 + 10u - 5 = c w^2$. Analysis of the constant factors restricts $c$ to the divisors of $5 \cdot 7 \cdot 11 \cdot 73$ that are congruent to $3 \pmod 8$. Multiplying by $27c^3$ and substituting $X = 3cu$ yields the elliptic curves $Y^2 = X^3 - 7cX^2 + 30c^2X - 45c^3$. We find all integral points subject to the required divisibility constraints.

**Case 2: $p \mid T_{10}(x)$**
Dividing by $T_{10}(x)$ leaves the linear factors $x(x+1)(2x+1) = c w^2$, where $c$ divides $168630$. Multiplying by $4c^3$ and applying a change of variables yields the elliptic curves $Y^2 = X^3 + 3cX^2 + 2c^2X$.

The script below calculates the integral points across both families of curves and evaluates valid integer $x$ candidates against $S_{10}(x)$ to isolate near-solutions to the generalized cannonball problem.

In [ ]:
# =============================================================================
# Elliptic Curve Search for k = 10 and n Even
# =============================================================================

var('x')

def S(k):
    """Returns S_k(x) as a polynomial in x using Faulhaber's formula."""
    return (1 / (k + 1)) * sum(
        binomial(k + 1, m) * ((-1)^m) * bernoulli(m) * x^(k - m + 1) 
        for m in [0..k]
    )

def find_k10_case1_candidates():
    """
    Finds valid x-values from the elliptic curves when p does not divide T_10.
    """
    valid_x = set()
    
    # c divides 5 * 7 * 11 * 73 and c = 3 (mod 8)
    divs_case1 = [d for d in divisors(5 * 7 * 11 * 73) if d % 8 == 3]
    
    f = 3 * x^3 - 7 * x^2 + 10 * x - 5
    g = expand(3^2 * f(x = x / 3))
    
    for c in divs_case1:
        gc = c^3 * expand(g(x = x / c))
        gccoefs = gc.coefficients(sparse=False)
        Ec = EllipticCurve([0, Integer(gccoefs[2]), 0, Integer(gccoefs[1]), Integer(gccoefs[0])])
        
        try:
            Ec.rank(algorithm='pari')
            pts = Ec.integral_points()
            
            for pt in pts:
                X, Y = pt[0], pt[1]
                if Y != 0 and Y % (3 * c^2) == 0 and X % (3 * c) == 0:
                    u = X / (3 * c)
                    if u > 0 and sqrt(4 * u + 1) in ZZ:
                        v = sqrt(4 * u + 1)
                        x_val = (v - 1) / 2
                        if x_val in ZZ:
                            valid_x.add(x_val)
                            
        except RuntimeError:
            print(f"Skipping c={c}: SageMath failed to compute rank.")
            
    return valid_x

def find_k10_case2_candidates():
    """
    Finds valid x-values from the elliptic curves when p divides T_10.
    """
    valid_x = set()
    
    # c divides 2 * 3 * 5 * 7 * 11 * 73 = 168630
    divs_case2 = divisors(168630)
    
    f2 = x * (x + 1) * (2 * x + 1)
    g2 = expand(2^2 * f2(x = x / 2))
    
    for c in divs_case2:
        hc = c^3 * expand(g2(x = x / c))
        hcoefs = hc.coefficients(sparse=False)
        Ec = EllipticCurve([0, Integer(hcoefs[2]), 0, Integer(hcoefs[1]), Integer(hcoefs[0])])
        
        try:
            Ec.rank(algorithm='pari', pari_effort=25)
            pts = Ec.integral_points()
        except RuntimeError:
            badcrank = Ec.rank(only_use_mwrank=False, algorithm='pari', pari_effort=25)
            pts = Ec.torsion_points() if badcrank == 0 else Ec.integral_points()
                
        for pt in pts:
            X, Y = pt[0], pt[1]
            if Y != 0 and Y % (2 * c^2) == 0 and X > 0 and X % (2 * c) == 0:
                x_val = X / (2 * c)
                if x_val in ZZ:
                    valid_x.add(x_val)
                    
    return valid_x

def verify_S10_solutions(candidate_x_values):
    """
    Evaluates S_10(x) for all candidate x-values to isolate near-solutions 
    of the form p^alpha * y^n.
    """
    print("\n" + "=" * 70)
    print("Verifying Candidates against S_10(x) = p * y^2")
    print("=" * 70)
    
    for x_val in sorted(list(candidate_x_values)):
        s_val = ZZ(S(10)(x=x_val))
        
        if s_val == 0:
            continue
            
        sqf_part = s_val.squarefree_part()
        
        if sqf_part.is_prime():
            y_val = sqrt(s_val / sqf_part)
            print(f"VALID SOLUTION: x = {str(x_val):<3} | y = {str(y_val):<5} | p = {sqf_part}")
        else:
            print(f"Dismissed: x = {str(x_val):<3} | Squarefree part ({sqf_part}) is not prime.")

# =============================================================================
# Execution Pipeline
# =============================================================================

print("Searching for integral points on elliptic curves for k = 10...")
print("-" * 70)

# Execute searches
candidates_case1 = find_k10_case1_candidates()
candidates_case2 = find_k10_case2_candidates()

# Combine unique candidates from both cases
all_candidates = candidates_case1.union(candidates_case2)

# Run verifications
if all_candidates:
    verify_S10_solutions(all_candidates)
else:
    print("\nNo integer candidates found.")

Searching for integral points on elliptic curves for k = 10...
----------------------------------------------------------------------

Verifying Candidates against S_10(x) = p * y^2
Dismissed: x = 1   | Squarefree part (1) is not prime.
VALID SOLUTION: x = 2   | y = 5     | p = 41
Dismissed: x = 3   | Squarefree part (1226) is not prime.
Dismissed: x = 4   | Squarefree part (44346) is not prime.
Dismissed: x = 5   | Squarefree part (434971) is not prime.
Dismissed: x = 7   | Squarefree part (3538157) is not prime.
Dismissed: x = 10  | Squarefree part (12174973) is not prime.
Dismissed: x = 24  | Squarefree part (35149646455) is not prime.
Dismissed: x = 27  | Squarefree part (2731817453414) is not prime.
Dismissed: x = 49  | Squarefree part (323829178517265) is not prime.
Dismissed: x = 840 | Squarefree part (95094562224443289135677) is not prime.
